<a href="https://colab.research.google.com/github/ekt4r/playground-series-s6e3/blob/main/lightgbm_tuned.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
train = pd.read_csv('/content/drive/MyDrive/playground-series-s6e3/train.csv').drop('id', axis=1)
test = pd.read_csv('/content/drive/MyDrive/playground-series-s6e3/test.csv').drop('id', axis=1)
sample_submission = pd.read_csv('/content/drive/MyDrive/playground-series-s6e3/sample_submission.csv')

In [4]:
for df in [train, test]:

    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df['TotalCharges'].fillna(df['MonthlyCharges'] * df['tenure'], inplace=True)

    # df['MCSquared'] = df['MonthlyCharges'] ** 2
    # df['TCSquared'] = df['TotalCharges'] ** 2

    df['MCLog'] = np.log1p(df['MonthlyCharges'])
    df['TCLog'] = np.log1p(df['TotalCharges'])

    # df['MCSqrt'] = np.sqrt(df['MonthlyCharges'])
    # df['TCSqrt'] = np.sqrt(df['TotalCharges'])

    df['MCSqueezed'] = 1 / (df['MonthlyCharges'] + 1)
    df['TCSqueezed'] = 1 / (df['TotalCharges'] + 1)

    # Loyalty
    df["TenureLog"] = np.log1p(df["tenure"])
    # df["TenureSquared"] = df["tenure"]**2
    # df['TenureSqrt'] = np.sqrt(df['tenure'])
    df['TenureSqueezed'] = 1 / (df['tenure'] + 1)

    # Spending
    df["AvgMonthlySpend"] = df["TotalCharges"] / (df["tenure"] + 1)
    df["SpendDeviation"] = df["MonthlyCharges"] - df["AvgMonthlySpend"]

    # Contract
    df["IsMonthToMonth"] = (df["Contract"] == "Month-to-month").astype(int)

    # Payment
    df["IsElectronicCheck"] = (df["PaymentMethod"] == "Electronic check").astype(int)
    df["AutoPayment"] = df["PaymentMethod"].str.contains("automatic").astype(int)

    # Service Engagement
    df["ServiceCount"] = (
        (df["OnlineSecurity"] == "Yes").astype(int) +
        (df["OnlineBackup"] == "Yes").astype(int) +
        (df["DeviceProtection"] == "Yes").astype(int) +
        (df["TechSupport"] == "Yes").astype(int) +
        (df["StreamingTV"] == "Yes").astype(int) +
        (df["StreamingMovies"] == "Yes").astype(int)
    )

    # Internet
    df["IsFiber"] = (df["InternetService"] == "Fiber optic").astype(int)

    # Early Risk
    df["EarlyTenure"] = (df["tenure"] < 12).astype(int)
    df["HighChargeEarly"] = (
        (df["tenure"] < 12) &
        (df["MonthlyCharges"] > df["MonthlyCharges"].median())
    ).astype(int)

    # Value
    df["LifetimeValue"] = df["MonthlyCharges"] * df["tenure"]

train['MCGT75%'] = train['MonthlyCharges'].gt(train['MonthlyCharges'].quantile(0.75)).astype(int)
test['MCGT75%'] = test['MonthlyCharges'].gt(train['MonthlyCharges'].quantile(0.75)).astype(int)

train['TCGT75%'] = train['TotalCharges'].gt(train['TotalCharges'].quantile(0.75)).astype(int)
test['TCGT75%'] = test['TotalCharges'].gt(train['TotalCharges'].quantile(0.75)).astype(int)

/tmp/ipykernel_1652/4109498660.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['TotalCharges'].fillna(df['MonthlyCharges'] * df['tenure'], inplace=True)
/tmp/ipykernel_1652/4109498660.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method

In [5]:
X, y = train.drop('Churn', axis=1), train['Churn']
y = y.map({'Yes': 1, 'No': 0})

In [6]:
from sklearn.preprocessing import OneHotEncoder, LabelEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

label_cols = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling'
]

onehot_cols = ['Contract', 'PaymentMethod']

ord_mapping = {
    'MultipleLines': ['No phone service', 'No', 'Yes'],
    'InternetService': ['No', 'DSL', 'Fiber optic'],
    'OnlineSecurity': ['No internet service', 'No', 'Yes'],
    'OnlineBackup': ['No internet service', 'No', 'Yes'],
    'DeviceProtection': ['No internet service', 'No', 'Yes'],
    'TechSupport': ['No internet service', 'No', 'Yes'],
    'StreamingTV': ['No internet service', 'No', 'Yes'],
    'StreamingMovies': ['No internet service', 'No', 'Yes'],
    'Partner': ['No', 'Yes'],
    'Dependents': ['No', 'Yes'],
    'PhoneService': ['No', 'Yes'],
    'PaperlessBilling': ['No', 'Yes'],
    'gender': ['Male', 'Female']
}

ord_cols = list(ord_mapping.keys())

preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', OneHotEncoder(drop='first', sparse_output=False), onehot_cols),
        ('ordinal', OrdinalEncoder(categories=[ord_mapping[col] for col in ord_cols]), ord_cols)
    ],
    remainder='passthrough'
)

pipeline = Pipeline([
    ('preprocessor', preprocessor)
])

X_train_encoded = pipeline.fit_transform(X)
X_test_encoded = pipeline.transform(test)

In [7]:
!nvidia-smi

Sat Mar  7 12:46:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [9]:
!pip install lightgbm optuna

In [ ]:
# Source - https://stackoverflow.com/a/14463362
# Posted by Mike, modified by community. See post 'Timeline' for change history
# Retrieved 2026-03-07, License - CC BY-SA 4.0

import warnings
warnings.filterwarnings("ignore")

In [14]:
import optuna
import lightgbm as lgb

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

def objective(trial):

    params = {
        "objective": "binary",
        "metric": "auc",
        "boosting_type": "gbdt",
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 16, 256),
        "max_depth": trial.suggest_int("max_depth", 1, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 20, 200),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 10.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 10.0),
        "n_estimators": 1000,
        "random_state": 42,
        "device": "gpu"
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    oof = np.zeros(len(X_train_encoded))

    for train_idx, valid_idx in skf.split(X_train_encoded, y):

        X_train, X_valid = X_train_encoded[train_idx], X_train_encoded[valid_idx]
        y_train, y_valid = y[train_idx], y[valid_idx]

        model = lgb.LGBMClassifier(**params, verbosity=-1)

        model.fit(
            X_train,
            y_train,
            eval_set=[(X_valid, y_valid)],
            eval_metric="auc",
            callbacks=[lgb.early_stopping(200), lgb.log_evaluation(0)]
        )

        preds = model.predict_proba(X_valid)[:, 1]
        oof[valid_idx] = preds

    score = roc_auc_score(y, oof)

    return score

In [15]:
study = optuna.create_study(
    direction="maximize"
)

study.optimize(objective, n_trials=50)

[I 2026-03-07 13:44:48,324] A new study created in memory with name: no-name-05482872-8c7e-4917-a910-562151990252


Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[358]	valid_0's auc: 0.915871


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Training until validation scores don't improve for 200 rounds
Early stopping, best iteration is:
[310]	valid_0's auc: 0.916865


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


...

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Training until validation scores don't improve for 200 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's auc: 0.917911


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Training until validation scores don't improve for 200 rounds
Did not meet early stopping. Best iteration is:
[995]	valid_0's auc: 0.915135


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-03-07 17:00:59,057] Trial 49 finished with value: 0.9167036078544161 and parameters: {'learning_rate': 0.09165993011011014, 'num_leaves': 178, 'max_depth': 3, 'min_child_samples': 188, 'subsample': 0.9102237034124121, 'colsample_bytree': 0.8015607032326315, 'reg_alpha': 4.67921007684637, 'reg_lambda': 9.106411151647325}. Best is trial 10 with value: 0.9168855489406124.


In [16]:
import json

with open('lightgbm-params.json', 'w') as json_file:
    json.dump(study.best_params, json_file, indent=4)

In [17]:
print(study.best_trial)
print(study.best_params)

FrozenTrial(number=10, state=<TrialState.COMPLETE: 1>, values=[0.9168855489406124], datetime_start=datetime.datetime(2026, 3, 7, 14, 32, 31, 769375), datetime_complete=datetime.datetime(2026, 3, 7, 14, 36, 41, 47283), params={'learning_rate': 0.16520956962166405, 'num_leaves': 136, 'max_depth': 3, 'min_child_samples': 136, 'subsample': 0.8855851541209333, 'colsample_bytree': 0.84199970908753, 'reg_alpha': 6.223316982700528, 'reg_lambda': 9.821692088402148}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'learning_rate': FloatDistribution(high=0.2, log=True, low=0.01, step=None), 'num_leaves': IntDistribution(high=256, log=False, low=16, step=1), 'max_depth': IntDistribution(high=12, log=False, low=1, step=1), 'min_child_samples': IntDistribution(high=200, log=False, low=20, step=1), 'subsample': FloatDistribution(high=1.0, log=False, low=0.6, step=None), 'colsample_bytree': FloatDistribution(high=1.0, log=False, low=0.6, step=None), 'reg_alpha': FloatDistributio

In [19]:
best_params = study.best_params

from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_preds = np.zeros(len(X_train_encoded))
test_preds = np.zeros(len(test))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X_train_encoded, y)):

    print(f'Fold {fold+1}')

    X_train, X_valid = X_train_encoded[train_idx], X_train_encoded[valid_idx]
    y_train, y_valid = y[train_idx], y[valid_idx]

    best_params = study.best_params

    model = lgb.LGBMClassifier(
        **best_params,
        objective="binary",
        metric="auc",
        n_estimators=1000
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_valid, y_valid)],
    )
    oof_preds[valid_idx] = model.predict_proba(X_valid)[:, 1]
    test_preds += model.predict_proba(X_test_encoded)[:, 1] / skf.n_splits

Fold 1


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 2


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 3


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 4


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Fold 5


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [20]:
cv_score = roc_auc_score(y, oof_preds)
print("CV ROC-AUC:", cv_score)

CV ROC-AUC: 0.9168624013531519


In [21]:
submission = sample_submission.copy()
submission['Churn'] = test_preds
submission.to_csv('submission.csv', index=False)

In [23]:
import os
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

In [24]:
!kaggle competitions submit -c playground-series-s6e3 -f submission.csv  -m "LightGBM CV5 Finetuned"

100% 6.60M/6.60M [00:00<00:00, 9.11MB/s]
Successfully submitted to Predict Customer Churn